# Exercise 1 - KNN on MNIST dataset

In [7]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tarfile
from io import BytesIO
from urllib.request import urlopen

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [2]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)

X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

knn = KNeighborsClassifier()
param_grid = {"n_neighbors":[3,4,5], "weights":["uniform","distance"]}

grid = GridSearchCV(knn, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))

Best parameters: {'n_neighbors': 3, 'weights': 'uniform'}
Best CV accuracy: nan
Test accuracy: 0.9705


# Exercise 2 - KNN on augmented MNIST dataset

In [3]:
def shift_images(X, dx, dy):
    images = X.reshape(-1, 28, 28)
    shifted = np.roll(images, shift=(dy, dx), axis=(1, 2))

    if dy > 0:
        shifted[:, :dy, :] = 0
    elif dy < 0:
        shifted[:, dy:, :] = 0

    if dx > 0:
        shifted[:, :, :dx] = 0
    elif dx < 0:
        shifted[:, :, dx:] = 0

    return shifted.reshape(-1, 784)

X_left  = shift_images(X_train, -1, 0)
X_right = shift_images(X_train, 1, 0)
X_up    = shift_images(X_train, 0, -1)
X_down  = shift_images(X_train, 0, 1)

X_train_aug = np.concatenate([X_train, X_left, X_right, X_up, X_down])
y_train_aug = np.tile(y_train, 5)

print("Original:", X_train.shape)
print("Augmented:", X_train_aug.shape)

Original: (60000, 784)
Augmented: (300000, 784)


In [4]:
best_knn.fit(X_train_aug, y_train_aug)
y_pred_aug = best_knn.predict(X_test)
print("Test accuracy after augmentation:", accuracy_score(y_test, y_pred_aug))

Test accuracy after augmentation: 0.9756


The accuracy slightly increases for the dataset with data augmented/training set explanded (4 extra images per image with each image shifted by one pixel in each direction).

# Exercise 3 - Classification on the Titanic dataset

In [8]:
url = "https://github.com/ageron/data/raw/main/titanic.tgz"

with tarfile.open(fileobj=BytesIO(urlopen(url).read()), mode="r:gz") as tar:
    train_data = pd.read_csv(tar.extractfile("titanic/train.csv"))
    test_data = pd.read_csv(tar.extractfile("titanic/test.csv"))

print("Training shape:", train_data.shape)
print("Test shape:", test_data.shape)

train_data.head()

Training shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
X_train = train_data.drop("Survived", axis=1)
y_train = train_data["Survived"]

num_attribs = ["Age", "SibSp", "Parch", "Fare"]
cat_attribs = ["Pclass", "Sex", "Embarked"]

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
cat_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
preprocessing = ColumnTransformer([("num", num_pipeline, num_attribs), ("cat", cat_pipeline, cat_attribs)])
model = make_pipeline(preprocessing, RandomForestClassifier(n_estimators=100, random_state=42))
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")

print("=" * 60)
print("TITANIC CLASSIFIER")
print("=" * 60)
print(f"CV accuracy: {scores.mean():.4f}")
print(f"CV std:      {scores.std():.4f}")

TITANIC CLASSIFIER
CV accuracy: 0.8059
CV std:      0.0326


A Random Forest classifier was trained on the Titanic dataset using a preprocessing pipeline to handle missing values, scale numerical features, and encode categorical features. Five-fold cross-validation was used to evaluate the model, providing an estimate of its generalization performance. We got a decent accuracy and a very small CV error suggesting model fits well on unseen data.

# Exercise 4 - SPAM classifier

In [22]:
import os
import re
import tarfile
import email
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from bs4 import BeautifulSoup
from email import policy
from email.parser import BytesParser

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [8]:
SPAM_PATH = Path("datasets/spam")
SPAM_PATH.mkdir(parents=True, exist_ok=True)

DOWNLOAD_ROOT = "https://spamassassin.apache.org/old/publiccorpus/"
HAM_URL = DOWNLOAD_ROOT + "20030228_easy_ham.tar.bz2"
SPAM_URL = DOWNLOAD_ROOT + "20030228_spam.tar.bz2"

def download_and_extract(url, filename):
    archive_path = SPAM_PATH / filename

    if not archive_path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, archive_path)

    extract_dir = SPAM_PATH / filename.replace(".tar.bz2", "")

    if not extract_dir.exists():
        print(f"Extracting {filename}...")
        with tarfile.open(archive_path, "r:bz2") as tar:
            tar.extractall(SPAM_PATH)

download_and_extract(HAM_URL, "ham.tar.bz2")
download_and_extract(SPAM_URL, "spam.tar.bz2")

Extracting ham.tar.bz2...


C:\Users\sumuk\AppData\Local\Temp\ipykernel_26268\991366643.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(SPAM_PATH)


In [9]:
ham_files = sorted((SPAM_PATH / "easy_ham").iterdir())
spam_files = sorted((SPAM_PATH / "spam").iterdir())

ham_files = [f for f in ham_files if f.is_file()]
spam_files = [f for f in spam_files if f.is_file()]

print("Ham emails:", len(ham_files))
print("Spam emails:", len(spam_files))

Ham emails: 2501
Spam emails: 501


In [10]:
with open(ham_files[0], "rb") as f:
    raw_email = f.read()

print(raw_email[:3000].decode("latin1"))

From exmh-workers-admin@redhat.com  Thu Aug 22 12:36:23 2002
Return-Path: <exmh-workers-admin@spamassassin.taint.org>
Delivered-To: zzzz@localhost.netnoteinc.com
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.netnoteinc.com (Postfix) with ESMTP id D03E543C36
	for <zzzz@localhost>; Thu, 22 Aug 2002 07:36:16 -0400 (EDT)
Received: from phobos [127.0.0.1]
	by localhost with IMAP (fetchmail-5.9.0)
	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 12:36:16 +0100 (IST)
Received: from listman.spamassassin.taint.org (listman.spamassassin.taint.org [66.187.233.211]) by
    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g7MBYrZ04811 for
    <zzzz-exmh@spamassassin.taint.org>; Thu, 22 Aug 2002 12:34:53 +0100
Received: from listman.spamassassin.taint.org (localhost.localdomain [127.0.0.1]) by
    listman.redhat.com (Postfix) with ESMTP id 8386540858; Thu, 22 Aug 2002
    07:35:02 -0400 (EDT)
Delivered-To: exmh-workers@listman.spamassassin.taint.org
Received: from int-mx1.corp

In [11]:
def get_email_body(path):
    with open(path, "rb") as f:
        msg = BytesParser(policy=policy.default).parse(f)

    parts = []
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()

            if content_type == "text/plain":
                try:
                    parts.append(part.get_content())
                except:
                    pass
            elif content_type == "text/html":
                try:
                    html = part.get_content()
                    soup = BeautifulSoup(html, "html.parser")
                    parts.append(soup.get_text(" ", strip=True))
                except:
                    pass
    else:
        try:
            content = msg.get_content()
            if msg.get_content_type() == "text/html":
                soup = BeautifulSoup(content, "html.parser")
                content = soup.get_text(" ", strip=True)
            parts.append(content)
        except:
            pass
    return "\n".join(parts)

In [12]:
emails = []
labels = []

for path in ham_files:
    try:
        emails.append(get_email_body(path))
        labels.append(0)
    except Exception:
        pass

for path in spam_files:
    try:
        emails.append(get_email_body(path))
        labels.append(1)
    except Exception:
        pass

emails = np.array(emails, dtype=object)
labels = np.array(labels)

print("Total emails:", len(emails))
print("Ham:", np.sum(labels == 0))
print("Spam:", np.sum(labels == 1))

Total emails: 3002
Ham: 2501
Spam: 501


In [13]:
X_train, X_test, y_train, y_test = train_test_split(emails, labels, test_size=0.2, random_state=42, stratify=labels)
print("Training:", len(X_train))
print("Test:", len(X_test))

Training: 2401
Test: 601


In [14]:
class EmailPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, lowercase=True, replace_urls=True, replace_numbers=True, remove_punctuation=True):
        self.lowercase = lowercase
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.remove_punctuation = remove_punctuation

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self._clean_email(email) for email in X]

    def _clean_email(self, text):
        text = str(text)

        if self.lowercase:
            text = text.lower()
        if self.replace_urls:
            text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
        if self.replace_numbers:
            text = re.sub(r"\b\d+\b", " NUMBER ", text)
        if self.remove_punctuation:
            text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()

        return text

In [15]:
preprocessor = EmailPreprocessor()
X_train_clean = preprocessor.fit_transform(X_train)
X_test_clean = preprocessor.transform(X_test)
vectorizer = CountVectorizer(stop_words="english", min_df=2)
X_train_vectorized = vectorizer.fit_transform(X_train_clean)
X_test_vectorized = vectorizer.transform(X_test_clean)

print("Training matrix:", X_train_vectorized.shape)
print("Test matrix:", X_test_vectorized.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Training matrix: (2401, 14670)
Test matrix: (601, 14670)
Vocabulary size: 14670


In [16]:
print(X_train_vectorized[0])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 58 stored elements and shape (1, 14670)>
  Coords	Values
  (0, 13217)	1
  (0, 7149)	1
  (0, 14529)	1
  (0, 874)	1
  (0, 9320)	3
  (0, 10725)	4
  (0, 5985)	4
  (0, 8924)	18
  (0, 7909)	1
  (0, 8762)	1
  (0, 10862)	2
  (0, 12193)	1
  (0, 8101)	1
  (0, 9052)	1
  (0, 7294)	1
  (0, 1451)	1
  (0, 13872)	1
  (0, 8402)	1
  (0, 14018)	1
  (0, 5658)	1
  (0, 7584)	1
  (0, 6575)	1
  (0, 7408)	1
  (0, 2083)	1
  (0, 11673)	1
  :	:
  (0, 7742)	1
  (0, 6567)	1
  (0, 3893)	1
  (0, 14444)	1
  (0, 2677)	1
  (0, 5833)	1
  (0, 7688)	1
  (0, 10932)	1
  (0, 1044)	1
  (0, 9932)	1
  (0, 8050)	1
  (0, 2423)	1
  (0, 3245)	1
  (0, 7654)	3
  (0, 11306)	3
  (0, 13847)	2
  (0, 13921)	1
  (0, 11350)	1
  (0, 7225)	1
  (0, 48)	1
  (0, 7693)	1
  (0, 7664)	3
  (0, 7879)	1
  (0, 5365)	1
  (0, 8742)	1


In [17]:
word_counts = np.asarray(X_train_vectorized.sum(axis=0)).ravel()
words = np.array(vectorizer.get_feature_names_out())
top_indices = word_counts.argsort()[-30:][::-1]
top_words = pd.DataFrame({"word": words[top_indices], "count": word_counts[top_indices]})
top_words

,word,count
0,number,33421
1,url,4782
2,mv,3001
3,list,1754
4,com,1253
5,just,1245
6,people,1109
7,net,1106
8,time,1063
9,new,1028


In [18]:
log_reg = LogisticRegression(max_iter=1000, C=10, n_jobs=-1)
log_reg.fit(X_train_vectorized, y_train)
y_pred_lr = log_reg.predict(X_test_vectorized)

print("Logistic Regression")
print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall   :", recall_score(y_test, y_pred_lr))
print("F1       :", f1_score(y_test, y_pred_lr))

Logistic Regression
Accuracy : 0.9850249584026622
Precision: 0.9690721649484536
Recall   : 0.94
F1       : 0.9543147208121827


In [19]:
nb = MultinomialNB()
nb.fit(X_train_vectorized, y_train)
y_pred_nb = nb.predict(X_test_vectorized)

print("Multinomial Naive Bayes")
print("Accuracy :", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall   :", recall_score(y_test, y_pred_nb))
print("F1       :", f1_score(y_test, y_pred_nb))

Multinomial Naive Bayes
Accuracy : 0.9833610648918469
Precision: 0.9787234042553191
Recall   : 0.92
F1       : 0.9484536082474226


In [20]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Multinomial Naive Bayes"],
    "Accuracy": [accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_nb)],
    "Precision": [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_nb)],
    "Recall": [recall_score(y_test, y_pred_lr), recall_score(y_test, y_pred_nb)],
    "F1": [f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_nb)]})
results

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.985025,0.969072,0.94,0.954315
1,Multinomial Naive Bayes,0.983361,0.978723,0.92,0.948454


In [21]:
best_model = log_reg
y_pred = y_pred_lr
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))
cv_scores = cross_val_score(log_reg, X_train_vectorized, y_train, cv=5, scoring="f1")

print("F1 scores:", cv_scores)
print("Mean F1:", cv_scores.mean())
print("F1 std:", cv_scores.std())

[[498   3]
 [  6  94]]
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       501
        Spam       0.97      0.94      0.95       100

    accuracy                           0.99       601
   macro avg       0.98      0.97      0.97       601
weighted avg       0.98      0.99      0.98       601

F1 scores: [0.91823899 0.92105263 0.95541401 0.95597484 0.92810458]
Mean F1: 0.9357570111918374
F1 std: 0.01659408278195594


In [23]:
tfidf = TfidfVectorizer(stop_words="english", min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf = tfidf.transform(X_test_clean)
tfidf_model = LogisticRegression(max_iter=1000, C=10)
tfidf_model.fit(X_train_tfidf, y_train)
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

print("TF-IDF + Logistic Regression")
print("Accuracy :", accuracy_score(y_test, y_pred_tfidf))
print("Precision:", precision_score(y_test, y_pred_tfidf))
print("Recall   :", recall_score(y_test, y_pred_tfidf))
print("F1       :", f1_score(y_test, y_pred_tfidf))

TF-IDF + Logistic Regression
Accuracy : 0.9883527454242929
Precision: 1.0
Recall   : 0.93
F1       : 0.9637305699481865


In [24]:
final_results = pd.DataFrame({
    "Model": ["Logistic Regression + Count", "Multinomial NB + Count", "Logistic Regression + TF-IDF"],
    "Accuracy": [accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_nb), accuracy_score(y_test, y_pred_tfidf)],
    "Precision": [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_nb), precision_score(y_test, y_pred_tfidf)],
    "Recall": [recall_score(y_test, y_pred_lr), recall_score(y_test, y_pred_nb), recall_score(y_test, y_pred_tfidf)],
    "F1": [f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_nb), f1_score(y_test, y_pred_tfidf)]})

final_results.sort_values("F1", ascending=False)

,Model,Accuracy,Precision,Recall,F1
2,Logistic Regression + TF-IDF,0.988353,1.000000,0.93,0.963731
0,Logistic Regression + Count,0.985025,0.969072,0.94,0.954315
1,Multinomial NB + Count,0.983361,0.978723,0.92,0.948454


In [27]:
examples = [
    "Congratulations! You have won a free prize. Click this URL now!",
    "Hi, just wanted to confirm our meeting tomorrow at 10am.",
    "URGENT! You have been selected to receive $5000. Click here!",
    "Thanks for sending the report. I will review it tonight."
]

examples_clean = preprocessor.transform(examples)
examples_vectorized = vectorizer.transform(examples_clean)

predictions = log_reg.predict(examples_vectorized)
probabilities = log_reg.predict_proba(examples_vectorized)[:, 1]

for email_text, prediction, probability in zip(examples, predictions, probabilities):
    print("-" * 70)
    print(email_text)
    print("Prediction:", "SPAM" if prediction else "HAM")
    print("Spam probability:", round(probability, 4))

----------------------------------------------------------------------
Congratulations! You have won a free prize. Click this URL now!
Prediction: SPAM
Spam probability: 0.9338
----------------------------------------------------------------------
Hi, just wanted to confirm our meeting tomorrow at 10am.
Prediction: SPAM
Spam probability: 0.5761
----------------------------------------------------------------------
URGENT! You have been selected to receive $5000. Click here!
Prediction: SPAM
Spam probability: 0.9926
----------------------------------------------------------------------
Thanks for sending the report. I will review it tonight.
Prediction: SPAM
Spam probability: 0.6579


- The exercise demonstrates how raw unstructured text can be transformed into numerical features suitable for machine learning and why precision and recall are particularly important when evaluating a spam classifier.
- The final results classify all emails as spam but the ones that contain important info have a lower probability. The before training results show a classic case of overfitting. Lowering the threshold for it to be classified as spam may help increase the accuracy on the test set.